# Tag NFC — record NDEF dan memori tag Type 2

Pembaca PC/SC virtual dengan tag NTAG213 simulasi. Dengan pembaca contactless sungguhan (ACR122U, ACR1252U…), ganti pembaca virtual dengan `PcscNfcReader.Open()`; sisa kodenya tetap sama.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.22.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.22.0-preview.1"

## Record NDEF
URI memakai kode prefiks satu byte (0x04 = https://), record Text membawa bahasanya, dan Smart Poster menampung keduanya.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.Nfc;

var ndef = new NdefMessage([NdefRecord.SmartPoster("https://docs.example.com/p7", "Pump P-0007 manual"), NdefRecord.Text("Asset P-0007")]);
var ndefBytes = ndef.Encode();
Console.WriteLine(Convert.ToHexString(ndefBytes));
foreach (var f in NdefMessage.Describe(ndefBytes)) Console.WriteLine($"{f.Name,-16} {f.Value}");
var wifi = NdefRecord.WifiCredential(new WifiCredential("Plant-Commissioning", "example-only-key"));
Console.WriteLine(wifi);   // the key is never shown

## Membaca tag
GET DATA mengembalikan UID, READ BINARY empat halaman sekaligus; area data dimulai di halaman 4 dengan TLV.

In [ ]:
var nfcReader = new VirtualNfcReader();
var pumpTag = new VirtualType2Tag(Type2TagKind.Ntag213, [0x04, 0x51, 0x7A, 0x22, 0x9C, 0x61, 0x80], ndef);
nfcReader.Present(pumpTag);
await using var card = await nfcReader.WaitForTagAsync();
var tagClient = new Type2TagClient(card);
Console.WriteLine($"UID {Convert.ToHexString(await tagClient.GetUidAsync())}");
var (cc, dataArea) = await tagClient.ReadDataAreaAsync();
Console.WriteLine($"CC {Convert.ToHexString(cc)}: {cc[2] * 8} bytes");
Console.WriteLine(await tagClient.ReadNdefAsync());

## Memori, halaman demi halaman

In [ ]:
var dump = await tagClient.DumpAsync();
var regions = Type2Tag.Map(dump, 144);
for (var page = 0; page < 12; page++) Console.WriteLine($"{page,2}  {Convert.ToHexString(dump, page * 4, 4)}  {regions[page * 4]}");

## Menulis harus diizinkan
Penulisan ditolak sampai diizinkan; hanya halaman yang berubah yang ditulis, dan header TLV NDEF ditulis terakhir.

In [ ]:
try { await tagClient.WriteNdefAsync(new NdefMessage([NdefRecord.Text("x")])); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
var tagWriter = new Type2TagClient(card, new NfcTagOptions().AllowWrites());
var before = pumpTag.PagesWritten;
await tagWriter.WriteNdefAsync(new NdefMessage([.. ndef.Records, NdefRecord.Text("Serviced 2026-10-09 · tech 14")]));
Console.WriteLine($"{pumpTag.PagesWritten - before} pages written");
Console.WriteLine(await tagClient.ReadNdefAsync());

## Lebih lanjut
`iotcom nfc readers`, `iotcom nfc read --sim --dump`, `iotcom nfc write --uri https://example.com --sim --allow-write`, `iotcom nfc decode <hex>`, *Tag aset NFC* di Gallery, dan sampel NfcTagReader. Lihat `docs/id/protocols/nfc.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*